In [1]:
import torch
from google.colab import drive
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
DATA_DIR = "/content/drive/MyDrive/microGPT-dataset"
TOKENIZER_DIR = "/content/drive/MyDrive/microGPT-tokenizer"

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
with open(f"{DATA_DIR}/metadata.json") as f:
    metadata = json.load(f)

VOCAB_SIZE = metadata["vocab_size"]
BLOCK_SIZE = metadata["block_size"]

print(DEVICE, VOCAB_SIZE, BLOCK_SIZE)

cuda 30000 256


In [6]:
!cp /content/drive/MyDrive/microGPT-dataset/*.bin /content/

# Load Training Batches

In [7]:
train_data = np.memmap(
    "/content/train.bin",
    dtype=np.uint16,
    mode="r"
)

val_data = np.memmap(
    "/content/val.bin",
    dtype=np.uint16,
    mode="r"
)

def get_batch(split, batch_size):
    data = train_data if split == "train" else val_data

    starts = torch.randint(
        0,
        len(data) - BLOCK_SIZE - 1,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            data[start:start + BLOCK_SIZE].astype(np.int64)
        )
        for start in starts
    ])

    y = torch.stack([
        torch.from_numpy(
            data[start + 1:start + BLOCK_SIZE + 1].astype(np.int64)
        )
        for start in starts
    ])

    return x.to(DEVICE), y.to(DEVICE)

In [8]:
x, y = get_batch("train", batch_size=4)

print(x.shape)  # [4, 256]
print(y.shape)  # [4, 256]

torch.Size([4, 256])
torch.Size([4, 256])


# Decoder-Only Transformer

In [9]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.projection = nn.Linear(n_embd, n_embd)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
            .view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        )

    def forward(self, x):
        batch_size, sequence_length, n_embd = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.split(n_embd, dim=2)

        q = q.view(
            batch_size,
            sequence_length,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            sequence_length,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            sequence_length,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        attention_scores = q @ k.transpose(-2, -1)
        attention_scores *= self.head_dim ** -0.5

        # Prevent each position from seeing future tokens.
        attention_scores = attention_scores.masked_fill(
            self.causal_mask[:, :, :sequence_length, :sequence_length] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = attention_weights @ v

        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, sequence_length, n_embd)

        return self.dropout(self.projection(output))

# Transformer block

In [10]:
class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()

        self.layer_norm_1 = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention(
            n_embd,
            n_head,
            dropout
        )

        self.layer_norm_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attention(self.layer_norm_1(x))
        x = x + self.mlp(self.layer_norm_2(x))
        return x

# Generative Pre-trained Transformer

In [11]:
class GPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        n_embd=384,
        n_head=6,
        n_layer=6,
        dropout=0.1,
    ):
        super().__init__()

        self.block_size = block_size

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding = nn.Embedding(
            block_size,
            n_embd
        )

        self.blocks = nn.Sequential(*[
            TransformerBlock(
                n_embd,
                n_head,
                dropout
            )
            for _ in range(n_layer)
        ])

        self.final_layer_norm = nn.LayerNorm(n_embd)

        self.language_model_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        # Weight tying reduces parameters and often improves training.
        self.language_model_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._initialize_weights)

    def _initialize_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        batch_size, sequence_length = input_ids.shape

        if sequence_length > self.block_size:
            raise ValueError("Sequence is longer than block size")

        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        )

        token_vectors = self.token_embedding(input_ids)
        position_vectors = self.position_embedding(positions)

        x = token_vectors + position_vectors
        x = self.blocks(x)
        x = self.final_layer_norm(x)

        logits = self.language_model_head(x)

        loss = None

        if targets is not None:
            batch_size, sequence_length, vocab_size = logits.shape

            loss = F.cross_entropy(
                logits.view(-1, vocab_size),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            context = input_ids[:, -self.block_size:]

            logits, _ = self(context)

            logits = logits[:, -1, :]
            logits = logits / temperature

            probabilities = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            input_ids = torch.cat(
                (input_ids, next_token),
                dim=1
            )

        return input_ids

# Model training

In [12]:
model = GPT(
    vocab_size=VOCAB_SIZE,
    block_size=BLOCK_SIZE,
    n_embd=384,
    n_head=6,
    n_layer=6,
    dropout=0.1,
).to(DEVICE)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Parameters: {number_of_parameters:,}")

Parameters: 22,265,856


# Adjusting Model weights based on errors

In [13]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

# Training and Validation

In [ ]:
BATCH_SIZE = 32
MAX_STEPS = 10000
EVAL_INTERVAL = 500
EVAL_STEPS = 50
CHECKPOINT_INTERVAL = 1000

def estimate_loss():
    model.eval()
    results = {}

    with torch.no_grad():
        for split in ["train", "val"]:
            losses = []

            for _ in range(EVAL_STEPS):
                x, y = get_batch(split, BATCH_SIZE)
                _, loss = model(x, y)
                losses.append(loss.item())

            results[split] = sum(losses) / len(losses)

    model.train()
    return results

In [ ]:
os.makedirs("/content/checkpoints", exist_ok=True)

for step in range(MAX_STEPS):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()

        print(
            f"step {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    x, y = get_batch("train", BATCH_SIZE)

    optimizer.zero_grad(set_to_none=True)

    _, loss = model(x, y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    if step % CHECKPOINT_INTERVAL == 0:
        checkpoint_path = f"/content/checkpoints/step_{step}.pt"

        torch.save(
            {
                "step": step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "model_config": {
                    "vocab_size": VOCAB_SIZE,
                    "block_size": BLOCK_SIZE,
                    "n_embd": 384,
                    "n_head": 6,
                    "n_layer": 6,
                    "dropout": 0.1,
                },
            },
            checkpoint_path
        )

        print(f"Saved checkpoint: {checkpoint_path}")

step 0: train loss 10.4032, val loss 10.4022
Saved checkpoint: /content/checkpoints/step_0.pt
step 500: train loss 6.2742, val loss 6.2851
step 1000: train loss 5.8519, val loss 5.8238
Saved checkpoint: /content/checkpoints/step_1000.pt
step 1500: train loss 5.5289, val loss 5.5467
step 2000: train loss 5.3558, val loss 5.3110
Saved checkpoint: /content/checkpoints/step_2000.pt
step 2500: train loss 5.2136, val loss 5.1618
step 3000: train loss 5.0944, val loss 5.0108
Saved checkpoint: /content/checkpoints/step_3000.pt
step 3500: train loss 4.9823, val loss 4.8930
step 4000: train loss 4.8525, val loss 4.8064
Saved checkpoint: /content/checkpoints/step_4000.pt
step 4500: train loss 4.7342, val loss 4.6621
step 5000: train loss 4.6875, val loss 4.5803
Saved checkpoint: /content/checkpoints/step_5000.pt
step 5500: train loss 4.6088, val loss 4.4984
step 6000: train loss 4.5466, val loss 4.4736
Saved checkpoint: /content/checkpoints/step_6000.pt
step 6500: train loss 4.5043, val loss 4.41

In [ ]:
!cp -r /content/checkpoints /content/drive/MyDrive/microGPT-model/

# Crash fallback

In [ ]:
CHECKPOINT_DIR = "/content/checkpoints"

checkpoint_path = f"{CHECKPOINT_DIR}/step_6000.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_step = checkpoint["step"] + 1

print(f"Resuming from step {start_step}")

In [ ]:
MAX_STEPS = 20000
BATCH_SIZE = 32
EVAL_INTERVAL = 500
CHECKPOINT_INTERVAL = 500
EVAL_STEPS = 50

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_loss = float("inf")

for step in range(start_step, MAX_STEPS + 1):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()

        print(
            f"step {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]

            torch.save(
                {
                    "step": step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                    "model_config": {
                        "vocab_size": VOCAB_SIZE,
                        "block_size": BLOCK_SIZE,
                        "n_embd": 384,
                        "n_head": 6,
                        "n_layer": 6,
                        "dropout": 0.1,
                    },
                },
                f"{CHECKPOINT_DIR}/best_model.pt",
            )

            print("Saved new best model")

    x, y = get_batch("train", BATCH_SIZE)

    optimizer.zero_grad(set_to_none=True)

    _, loss = model(x, y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()

    if step % CHECKPOINT_INTERVAL == 0:
        torch.save(
            {
                "step": step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "model_config": {
                    "vocab_size": VOCAB_SIZE,
                    "block_size": BLOCK_SIZE,
                    "n_embd": 384,
                    "n_head": 6,
                    "n_layer": 6,
                    "dropout": 0.1,
                },
            },
            f"{CHECKPOINT_DIR}/step_{step}.pt",
        )

        print(f"Saved checkpoint at step {step}")

In [ ]:
final_model_path = f"{DRIVE_DIR}/final_model.pt"

torch.save(
    {
        "step": step,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "vocab_size": VOCAB_SIZE,
            "block_size": BLOCK_SIZE,
            "n_embd": 384,
            "n_head": 6,
            "n_layer": 6,
            "dropout": 0.1,
        },
    },
    final_model_path,
)

print(f"Final model saved to: {final_model_path}")

# Casual Testing

In [ ]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    "/content/drive/MyDrive/microGPT/tokenizer/vocab.json",
    "/content/drive/MyDrive/microGPT/tokenizer/merges.txt"
)

In [ ]:
prompt = "Alexander the great was a "

encoded_prompt = tokenizer.encode(prompt)
prompt_ids = torch.tensor(
    [encoded_prompt.ids],
    dtype=torch.long,
    device=DEVICE
)

generated_ids = model.generate(
    prompt_ids,
    max_new_tokens=100,
    temperature=0.8
)

generated_text = tokenizer.decode(
    generated_ids[0].tolist()
)

print(generated_text)